In [3]:
import os
import xarray as xr
import numpy as np

In [4]:
# === Processing function ===
def calculate_maximum_6month_mean(start_year, end_year, monthly_mda8):
    years = list(range(start_year, end_year + 1))

    max_vals = []

    for year in years:
        # Define window: Jan of this year to Mar of next year
        start = f"{year}-01"
        end = f"{year + 1}-03"

        # Subset to this window
        subset = monthly_mda8.sel(time=slice(start, end))

        # Compute 6-month rolling mean along time
        rolling_6m = subset.rolling(time=6, center=False).mean()

        # Find index of maximum
        max_idx = rolling_6m.argmax(dim="time")
        max_val = rolling_6m.isel(time=max_idx)

        # Expand dimensions for consistent output
        max_val = max_val.expand_dims(year=[year])

        max_vals.append(max_val)

    # Combine across years
    annual_max_6m = xr.concat(max_vals, dim="year")

    return annual_max_6m

In [8]:
# === Path config ===
BASE_DIR = "/glade/work/awells/air_quality/CESM/MDA8/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/OSDMA8_March/"
SCENARIOS = ["hist"]
ens_num = 1


# === Main loop ===
for scenario in SCENARIOS:
    print(f"Processing {scenario}")
    if scenario == "ARISE":
        dates = "20350101-20691231"
    elif scenario == "SSP245":
        dates = "20200101-20691231"
    elif scenario == "hist":
        dates = "19900101-20091231"
    file_list = [f"{BASE_DIR}MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"]
    OSDMA8 = []

    for file in file_list:
        if not os.path.exists(file):
            print(f"Missing: {file}")
            continue

        print(f"Reading {os.path.basename(file)}")
        monthly_mda8 = xr.open_dataarray(file)

        # Create list of years to calculate over
        start_year = int(str(monthly_mda8.time.dt.year[0].values))
        end_year = int(str(monthly_mda8.time.dt.year[-1].values))  # final year will be 12 months rather than 15

        annual_max_6m = calculate_maximum_6month_mean(start_year, end_year, monthly_mda8)

        OSDMA8.append(annual_max_6m)

    if OSDMA8:
        combined = xr.concat(OSDMA8, dim="year")

        out_file = f"OSDMA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        combined.to_netcdf(out_path)

print("All processing complete.")

Processing hist
Reading MDA8_CESM2_hist_01_19900101-20091231.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/OSDMA8_CESM2_hist_01_19900101-20091231.nc
All processing complete.


In [6]:
# === Path config ===
BASE_DIR = "/glade/work/awells/air_quality/CESM/MDA8/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/OSDMA8_March/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "20350101-20691231"
        else:
            dates = "20200101-20691231"
        file_list = [f"{BASE_DIR}MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"]
        OSDMA8 = []

        for file in file_list:
            if not os.path.exists(file):
                print(f"Missing: {file}")
                continue

            print(f"Reading {os.path.basename(file)}")
            monthly_mda8 = xr.open_dataarray(file)

            # Create list of years to calculate over
            start_year = int(str(monthly_mda8.time.dt.year[0].values))
            end_year = int(str(monthly_mda8.time.dt.year[-1].values))  # final year will be 12 months rather than 15

            annual_max_6m, max_start_times = calculate_maximum_6month_mean(start_year, end_year, monthly_mda8)

            OSDMA8.append(annual_max_6m)

        if OSDMA8:
            combined = xr.concat(OSDMA8, dim="year")

            out_file = f"OSDMA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            combined.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 01
Reading MDA8_CESM2_ARISE_01_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/OSDMA8_CESM2_ARISE_01_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_March/OSDMA8_startmonth_CESM2_ARISE_01_20350101-20691231.nc
All processing complete.
